In [1]:
# 주피터 노트북의 맨 첫 셀에 입력 (앞에 % 기호가 붙어야 합니다)
%env CUDA_HOME=/usr
%env NVCC_APPEND_FLAGS=--override

env: CUDA_HOME=/usr
env: NVCC_APPEND_FLAGS=--override


---
## 1️⃣ HuggingFace 로그인

**이론 요약**
- EXAONE 4.0도 Access Request 필요
- `.env`의 `HF_TOKEN` 사용

> ⚠️ 사전 준비: [EXAONE-4.0-1.2B](https://huggingface.co/LGAI-EXAONE/EXAONE-4.0-1.2B) 페이지에서 Access Request 승인받기.

In [2]:
from dotenv import load_dotenv
from huggingface_hub import login
import os

load_dotenv()

token = os.getenv("HF_TOKEN")
if not token:
    raise ValueError("❌ .env에 HF_TOKEN이 없습니다")

login(token=token)
print(f"✅ HuggingFace 로그인 완료")

/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HuggingFace 로그인 완료


---
## 2️⃣ CONFIG 설정

**이론 요약**
- EXAONE 4.0 1.2B는 hidden_size 2048 (3.5 2.4B는 2560)
- r=16이면 적절 (모델이 작아서 r=32는 과함)
- alpha = r × 2 = 32

In [3]:
CONFIG = {
    "model_name": "LGAI-EXAONE/EXAONE-4.0-1.2B",
    "max_seq_length": 1024,
    "lora_r": 16,           # EXAONE 4.0 1.2B에 맞게 조정
    "lora_alpha": 32,       # r × 2
    "lora_dropout": 0.05,
    "seed": 42,
}

print("📋 CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

📋 CONFIG:
  model_name: LGAI-EXAONE/EXAONE-4.0-1.2B
  max_seq_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  seed: 42


---
## 3️⃣ 토크나이저 + 양자화 설정

**이론 요약**
- EXAONE 4.0은 `trust_remote_code` **불필요** (공식 지원!)
- `compute_dtype=bfloat16` 권장 (RTX 3080 네이티브 지원)
- 패딩 설정은 동일

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

print(f"✅ 토크나이저 로드 완료")
print(f"   어휘 크기: {tokenizer.vocab_size:,}")
print(f"   pad_token: {tokenizer.pad_token}")

✅ 토크나이저 로드 완료
   어휘 크기: 102,400
   pad_token: [PAD]


In [5]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,   # EXAONE 4.0은 bfloat16 권장
    bnb_4bit_use_double_quant=True,
)

print("✅ 양자화 설정 완료 (compute_dtype=bfloat16)")

✅ 양자화 설정 완료 (compute_dtype=bfloat16)


---
## 4️⃣ EXAONE 4.0 1.2B 로드 (패치 없음!)

**이론 요약**
- EXAONE 3.5에서 필요했던 `get_input_embeddings` 패치 **불필요**
- `attn_implementation="eager"` **불필요**
- `trust_remote_code=True` **불필요**
- **HF 공식 아키텍처라 그냥 로드하면 끝**

```
첫 다운로드 예상:
  1.2B 모델 = 약 2.5GB
  다운로드: 2~4분 (네트워크에 따라)
  4-bit 양자화 후 VRAM: ~1GB

In [6]:
# 모델 로드
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training

print("🔄 EXAONE 4.0 1.2B 4-bit 로드 중... (2~4분)")

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    # trust_remote_code 없음! attn_implementation 없음!
)

model = prepare_model_for_kbit_training(model)

print(f"\n✅ 모델 로드 완료")


🔄 EXAONE 4.0 1.2B 4-bit 로드 중... (2~4분)


/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



✅ 모델 로드 완료


---
## 5️⃣ 모델 구조 확인 (target_modules 결정용)

**이론 요약**
- EXAONE 4.0의 실제 레이어 이름 확인
- Llama 계열 구조 → `q_proj, k_proj, v_proj, o_proj`
- Full 버전(3.5)의 `out_proj`와 다름

In [7]:
linear_modules = set()
for name, module in model.named_modules():
    if "Linear" in type(module).__name__ or "4bit" in type(module).__name__:
        if len(name.split(".")) >= 2:
            linear_modules.add(name.split(".")[-1])

print("📋 EXAONE 4.0 1.2B의 Linear 레이어 이름:")
for 이름 in sorted(linear_modules):
    print(f"  - {이름}")

📋 EXAONE 4.0 1.2B의 Linear 레이어 이름:
  - down_proj
  - gate_proj
  - k_proj
  - o_proj
  - q_proj
  - up_proj
  - v_proj


---
## 6️⃣ LoRA 어댑터 부착 → QLoRA 완성

**이론 요약**
- EXAONE 4.0은 Llama 계열 구조 → `q_proj, k_proj, v_proj, o_proj`
- MLP까지 포함: `gate_proj, up_proj, down_proj`
- r=16 (1.2B 모델에 적절)

In [8]:
# LoraConfig + get_peft_model

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",      # Attention
        "gate_proj", "up_proj", "down_proj",         # FFN
    ],
)

model = get_peft_model(model, lora_config)
print("🚀 QLoRA 완성!")

🚀 QLoRA 완성!


In [9]:
# 학습 파라미터 확인
model.print_trainable_parameters()

trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


In [10]:
# 메모리 확인
# print_gpu_memory("QLoRA 완성 후")

---
## 7️⃣ 베이스라인 응답 수집

**이론 요약**
- EXAONE 4.0은 `apply_chat_template` 표준 방식 사용
- 3.5의 `[|system|]...[|endofturn|]` 포맷과 다름
- HuggingFace 공식 채팅 템플릿 자동 적용

> 💡 **EXAONE 4.0 1.2B 권장 파라미터** (공식 문서):  
> - `temperature=0.1` (한국어 코드 스위칭 방지)  
> - `top_p=0.9`

In [11]:
# 추론 함수
def generate_fast(model, tokenizer, question, max_new_tokens=300):
    messages = [
        {"role": "system", "content": "당신은 대한민국 법률 전문 AI 어시스턴트입니다. 관련 법조항과 법률 원칙에 근거하여 정확한 답변을 제공합니다."},
        {"role": "user", "content": question}
    ]
    
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.inference_mode():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.1,            # EXAONE 4.0 1.2B 한국어 권장값
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 입력 부분 제외하고 디코딩
    생성 = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(생성, skip_special_tokens=True).strip()

print("✅ 추론 함수 정의 완료")

✅ 추론 함수 정의 완료


In [12]:
TEST_QUESTIONS = [
    "임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?",
    "근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?",
    "온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무엇인가요?",
    "교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?",
    "이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무엇인가요?",
]

print(f"[질문] {TEST_QUESTIONS[0]}\n")
응답 = generate_fast(model, tokenizer, TEST_QUESTIONS[0])
print(f"[응답]\n{응답}")

[질문] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?



/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[응답]
임차인이 보증금을 돌려받지 못할 경우 다음과 같은 법적 절차와 해결 방안을 고려할 수 있습니다. 관련 법조항과 절차를 중심으로 설명드리겠습니다.

### 1. **임대차 계약 해지 및 반환 청구 (민법 제618조)**
   - 임대인이 계약 해지 사유(예: 임차인의 계약 위반, 임대인의 귀책사유 없음 등)를 입증하면 임차인은 보증금 반환청구권을 행사할 수 있습니다.
   - 단, 해지 통지 기간(보통 3개월) 내에 반환되지 않으면 추가 소송 가능성 있음.

### 2. **보증금 반환채권 보전 조치 (민사집행법 제266조)**
   - 법원에 **채권보전 신청**을 할 경우, 압류·가압류 등을 통해 우선변제권을 확보할 수 있습니다.
   - 필요 시 임대차보호법에 따른 **임대차보호법 제8조(보증금 반환채권)**를 활용해 최우선변제권을 주장할 수 있습니다.

### 3. **임대인의 귀책사유 책임 (민법 제752조)**
   - 임대인이 계약상 의무 위반(예: 수리 neglected, 임차인 과실로 인한 손해)이 있을 경우, 손해배상 청구가 가능합니다.
   - 특히 주택임대차보호법 제6


In [13]:
print("🔄 베이스라인 응답 생성 중...\n")

baseline_results = []
for i, 질문 in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] {질문[:30]}...")
    응답 = generate_fast(model, tokenizer, 질문)
    baseline_results.append({
        "question": 질문,
        "baseline_answer": 응답,
    })

print(f"\n✅ {len(baseline_results)}건 수집 완료")

🔄 베이스라인 응답 생성 중...

[1/5] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요...
[2/5] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해...
[3/5] 온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무...
[4/5] 교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?...
[5/5] 이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무...

✅ 5건 수집 완료


## SFTTrainer 학습 실행 (간소화 버전)
### 약 3~5분 안에 파인튜닝 체험하기

> **환경**: RTX 3080 + Part 3 완료 상태  
> **학습 규모**: 200건 × 2 에폭 = **약 50 스텝**  
> **소요 시간**: 3~5분

---

## 🎯 5단계

| # | 내용 |
|---|------|
| 1 | 데이터 로드 (200건) |
| 2 | 채팅 템플릿 적용 + 토크나이징 |
| 3 | TrainingArguments 설정 |
| 4 | Trainer.train() 실행 🚀 |
| 5 | 학습 후 응답 비교 + 저장 |


---
## 1️⃣ 데이터 로드 (200건)

In [14]:
from datasets import load_dataset
import random

ds = load_dataset("jihye-moon/LawQA-Ko", split="train") # 한국어 법률 QA 데이터셋의 저장소 이름

# 컬럼 이름 확인
q_col, a_col = ds.column_names[:2]
print(f"컬럼: 질문='{q_col}', 답변='{a_col}'")

# 200건 샘플링
random.seed(42)
ds = ds.select(random.sample(range(len(ds)), 200))
print(f"✅ 학습 데이터: {len(ds)}건")

컬럼: 질문='question', 답변='precedent'
✅ 학습 데이터: 200건


---
## 2️⃣ 포맷 + 토크나이징

In [15]:
SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess(sample):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": sample[q_col]},
        {"role": "assistant", "content": sample[a_col]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

tokenized = ds.map(preprocess, remove_columns=ds.column_names)
print(f"✅ 토크나이징 완료: {len(tokenized)}건")

✅ 토크나이징 완료: 200건


---
## 3️⃣ TrainingArguments (50스텝 설정)

In [16]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-quick",
    num_train_epochs=2,                        # 2 에폭
    per_device_train_batch_size=2,             # 배치 2
    gradient_accumulation_steps=4,             # 실효 배치 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",                        # 체크포인트 저장 안함 (속도)
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)
예상스텝 = len(tokenized) // 8 * 2
print(f"✅ 예상 스텝: 약 {예상스텝}")

✅ 예상 스텝: 약 50


---
## 4️⃣ 학습 실행 🚀 (3~5분)

In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)

print("🏋️ 학습 시작!\n")
result = trainer.train()

print(f"\n✅ 학습 완료!")
print(f"   train_loss: {result.training_loss:.4f}")
print(f"   소요 시간:  {result.metrics['train_runtime']:.0f}초")

/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


🏋️ 학습 시작!



/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,3.894800
10,2.880100
15,2.609900
20,2.366500
25,2.518000
30,2.286500
35,2.496400
40,2.144300
45,2.021700
50,2.109300



✅ 학습 완료!
   train_loss: 2.5327
   소요 시간:  98초


---
## 5️⃣ 학습 후 응답 + 저장

In [18]:
model.eval()

# 첫 질문으로 Before/After 비교
질문 = TEST_QUESTIONS[0]
학습후 = generate_fast(model, tokenizer, 질문)
베이스라인 = baseline_results[0]["baseline_answer"]

print(f"[질문] {질문}\n")
print("=" * 55)
print("[학습 전]")
print("=" * 55)
print(베이스라인[:300])
print("\n" + "=" * 55)
print("[학습 후]")
print("=" * 55)
print(학습후[:300])

/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[질문] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?

[학습 전]
임차인이 보증금을 돌려받지 못할 경우 다음과 같은 법적 절차와 해결 방안을 고려할 수 있습니다. 관련 법조항과 절차를 중심으로 설명드리겠습니다.

### 1. **임대차 계약 해지 및 반환 청구 (민법 제618조)**
   - 임대인이 계약 해지 사유(예: 임차인의 계약 위반, 임대인의 귀책사유 없음 등)를 증명하면 임차인은 보증금 반환청구권을 행사할 수 있습니다.
   - 단, 해지 통지 기간(보통 3개월) 내에 반환되지 않으면 추가 소송 가능.

### 2. **보증금 반환채권 보전 조치 (민법 제323조)**
   - 임대인이

[학습 후]
임차인이 보증금을 돌려받지 못할 경우, 다음과 같은 방법을 고려해 보세요.


In [19]:
# 5개 질문 전체 수집
finetuned_results = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    finetuned_results.append({
        "question": q,
        "baseline": baseline_results[i-1]["baseline_answer"],
        "finetuned_raw": generate_fast(model, tokenizer, q),
    })

print(f"\n✅ {len(finetuned_results)}건 완료")

[1/5] 생성 중...
[2/5] 생성 중...
[3/5] 생성 중...
[4/5] 생성 중...
[5/5] 생성 중...

✅ 5건 완료


In [20]:
# 어댑터 + 결과 저장
import json, os

trainer.save_model("./exaone4-law-raw")
tokenizer.save_pretrained("./exaone4-law-raw")

os.makedirs("./outputs", exist_ok=True)
with open("./outputs/finetuned_raw_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss": result.training_loss,
        "results": finetuned_results,
    }, f, ensure_ascii=False, indent=2)

print("✅ 어댑터 저장: ./exaone4-law-raw/")
print("✅ 결과 저장:   ./outputs/finetuned_raw_results.json")

✅ 어댑터 저장: ./exaone4-law-raw/
✅ 결과 저장:   ./outputs/finetuned_raw_results.json


# 원본 vs 정제 데이터 비교 파인튜닝
### 데이터 품질이 모델 성능을 결정한다

> **환경**: RTX 3080 + Part 4 완료 상태  
> **소요 시간**: 정제 5분 + 학습 5분 + 평가 3분 = **약 13분**

---

## 🎯 5단계

| # | 내용 | 시간 |
|---|------|------|
| 1 | GPT-4o-mini로 200건 정제 | ~5분 |
| 2 | 정제 데이터로 재학습 (50스텝) | ~5분 |
| 3 | 3가지 응답 나란히 비교 | ~2분 |
| 4 | LLM Judge 채점 | ~1분 |
| 5 | 최종 비교표 + 저장 | |

---
## 1️⃣ GPT-4o-mini로 데이터 정제

**정제 규칙**
- 구체적 인명·지명·날짜·금액 제거
- 법률 원칙을 묻는 일반적 질문으로
- 답변에 법조항 명시 + 300~500자

In [21]:
from openai import OpenAI
from datasets import load_dataset
import json, random

client = OpenAI()  # .env의 OPENAI_API_KEY 자동 사용

# 원본 데이터 로드
ds_orig = load_dataset("jihye-moon/LawQA-Ko", split="train")
q_col, a_col = ds_orig.column_names[:2]

random.seed(42)
indices = random.sample(range(len(ds_orig)), 200)

print(f"✅ 정제할 샘플: {len(indices)}건")

✅ 정제할 샘플: 200건


In [22]:
def refine(question, answer):
    prompt = f"""당신은 대한민국 법률 교육 전문가입니다.
아래 법률 상담 QA를 **법률 원칙 학습용 데이터**로 정제해주세요.

[원본 질문]
{question[:600]}

[원본 답변]
{answer[:1000]}

[정제 규칙]
1. 질문에서 구체적 인명, 지명, 날짜, 금액을 제거하고 법률 원칙을 묻는 일반적 질문으로.
2. 답변에서 특정 사건 사실관계를 제거하고 법조항·법률 원칙 중심으로 재작성.
3. 답변은 300~500자. 반드시 한국어.

JSON 형식으로만 출력: {{"question": "...", "answer": "..."}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=800, temperature=0.3,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 1건 테스트
s = ds_orig[indices[0]]
test_result = refine(s[q_col], s[a_col])
print("📋 정제 테스트:")
print(f"  원본 Q: {s[q_col][:100]}...")
print(f"  정제 Q: {test_result['question'] if test_result else '실패'}")

📋 정제 테스트:
  원본 Q: 주택 임대차 계약을 체결하려고 하는데, 계약 당일에 주택의 소유자가 바빠서 소유자의 부인이랑 임대차 계약을 체결하라고 합니다. 부인이랑 임대차 계약을 체결해도 괜찮을까요?...
  정제 Q: 주택 임대차 계약을 체결할 때, 소유자가 아닌 제3자와 계약을 체결해도 법적으로 문제가 없는지 궁금합니다.


In [23]:
import time

# 200건 일괄 정제 (약 5분)
refined = []
failed = 0

print("🔄 정제 시작...\n")
for i, idx in enumerate(indices):
    s = ds_orig[idx]
    r = refine(s[q_col], s[a_col])
    
    if r and r.get("question") and r.get("answer"):
        refined.append({"instruction": r["question"], "output": r["answer"]})
    else:
        failed += 1
    
    if (i + 1) % 20 == 0:
        print(f"  [{i+1}/200] 성공 {len(refined)} / 실패 {failed}")
    time.sleep(0.3)

print(f"\n✅ 정제 완료: {len(refined)}건")

🔄 정제 시작...

  [20/200] 성공 20 / 실패 0
  [40/200] 성공 40 / 실패 0
  [60/200] 성공 60 / 실패 0
  [80/200] 성공 80 / 실패 0
  [100/200] 성공 100 / 실패 0
  [120/200] 성공 120 / 실패 0
  [140/200] 성공 140 / 실패 0
  [160/200] 성공 160 / 실패 0
  [180/200] 성공 180 / 실패 0
  [200/200] 성공 200 / 실패 0

✅ 정제 완료: 200건


In [24]:
# 정제 데이터 저장
import os
os.makedirs("./outputs", exist_ok=True)

with open("./outputs/refined_data.json", "w", encoding="utf-8") as f:
    json.dump(refined, f, ensure_ascii=False, indent=2)

print(f"💾 저장: ./outputs/refined_data.json ({len(refined)}건)")
print(f"\n📋 정제 샘플 2개:")
for i, d in enumerate(refined[:2]):
    print(f"\n[{i+1}] Q: {d['instruction'][:100]}")
    print(f"    A: {d['output'][:150]}...")

💾 저장: ./outputs/refined_data.json (200건)

📋 정제 샘플 2개:

[1] Q: 주택 임대차 계약을 체결할 때, 소유자가 아닌 제3자와 계약을 체결해도 되는지에 대한 법적 원칙은 무엇인가요?
    A: 주택 임대차 계약은 원칙적으로 임대인과 임차인 간의 합의에 의해 체결됩니다. 임대인은 주택의 소유자로서 계약을 체결할 권한을 가지고 있으며, 소유자가 아닌 제3자와 계약을 체결하는 경우에는 법적 효력이 제한될 수 있습니다. 만약 소유자가 부재하거나 계약을 체결할 수 없...

[2] Q: 자동차를 운전 중 사고가 발생했을 때, 운전자가 아닌 차량 소유자에게도 책임이 인정될 수 있는지에 대한 법적 원칙은 무엇인가요?
    A: 자동차손해배상보장법에 따르면, 자동차의 운행 중 발생한 사고에 대해 '운행자'의 개념이 중요합니다. 일반적으로 운행자는 차량을 직접 운전하는 자를 의미하지만, 특정 상황에서는 차량 소유자에게도 책임이 부과될 수 있습니다. 예를 들어, 차량 소유자가 운전자를 지명하거나 ...


---
## 2️⃣ 정제 데이터로 재학습

**주의**: `model`은 Part 4에서 이미 원본 데이터로 학습된 상태.  
새 어댑터로 다시 시작하기 위해 **Part 3 상태로 초기화** 필요.

In [25]:
# 기존 모델 해제
import gc
del model, trainer
gc.collect()
import torch
torch.cuda.empty_cache()

print("✅ Part 4 model 해제 완료")
print(f"현재 VRAM: {torch.cuda.memory_allocated()/1e9:.2f}GB")

✅ Part 4 model 해제 완료
현재 VRAM: 0.86GB


In [26]:
# EXAONE 4.0 + QLoRA 다시 로드
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

print("🔄 EXAONE 4.0 재로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"], quantization_config=bnb_config, device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
model = get_peft_model(model, lora_config)
print("✅ QLoRA 재적용 완료")
model.print_trainable_parameters()

🔄 EXAONE 4.0 재로드 중...


/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


✅ QLoRA 재적용 완료
trainable params: 15,237,120 || all params: 1,294,628,608 || trainable%: 1.1769


In [27]:
# 정제 데이터로 Dataset 변환 + 토크나이징
from datasets import Dataset

SYSTEM = "당신은 대한민국 법률 전문 AI 어시스턴트입니다."

def preprocess_refined(s):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": s["instruction"]},
        {"role": "assistant", "content": s["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    tok = tokenizer(text, truncation=True, max_length=1024, padding=False)
    tok["labels"] = tok["input_ids"].copy()
    return tok

ds_refined = Dataset.from_list(refined)
tokenized_refined = ds_refined.map(preprocess_refined, remove_columns=ds_refined.column_names)
print(f"✅ 토크나이징: {len(tokenized_refined)}건")

Map: 100%|██████████| 200/200 [00:00<00:00, 1319.27 examples/s]

✅ 토크나이징: 200건


In [28]:
# 학습 실행 (50스텝)
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

args = TrainingArguments(
    output_dir="./exaone4-law-refined-tmp",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    bf16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, pad_to_multiple_of=8)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_refined, data_collator=collator)

print("🏋️ 정제 데이터 학습 시작!\n")
result_refined = trainer.train()
print(f"\n✅ 완료! train_loss: {result_refined.training_loss:.4f}")

🏋️ 정제 데이터 학습 시작!



/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.388700
10,1.506200
15,1.257600
20,1.176800
25,1.192700
30,1.008000
35,0.996000
40,0.995800
45,0.958900
50,0.938300



✅ 완료! train_loss: 1.2419


---
## 3️⃣ 3가지 응답 나란히 비교

In [29]:
# 정제 학습본으로 5개 질문 응답 생성
model.eval()

refined_answers = []
for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"[{i}/5] 생성 중...")
    refined_answers.append(generate_fast(model, tokenizer, q))

print("✅ 정제 학습본 응답 5건 완료")

[1/5] 생성 중...


/home/ubuntu/llm_project/venv/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[2/5] 생성 중...
[3/5] 생성 중...
[4/5] 생성 중...
[5/5] 생성 중...
✅ 정제 학습본 응답 5건 완료


In [30]:
# 3가지 비교: 베이스라인 vs 원본학습 vs 정제학습
질문 = TEST_QUESTIONS[0]
베이스라인 = baseline_results[0]["baseline_answer"]
원본학습 = finetuned_results[0]["finetuned_raw"]
정제학습 = refined_answers[0]

print(f"[질문] {질문}\n")
for 이름, 응답 in [("1️⃣ 베이스라인 (학습 없음)", 베이스라인),
                 ("2️⃣ 원본 데이터 학습", 원본학습),
                 ("3️⃣ 정제 데이터 학습", 정제학습)]:
    print("=" * 55)
    print(이름)
    print("=" * 55)
    print(응답[:300])
    print()

[질문] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요?

1️⃣ 베이스라인 (학습 없음)
임차인이 보증금을 돌려받지 못할 경우 다음과 같은 법적 절차와 해결 방안을 고려할 수 있습니다. 관련 법조항과 절차를 중심으로 설명드리겠습니다.

### 1. **임대차 계약 해지 및 반환 청구 (민법 제618조)**
   - 임대인이 계약 해지 사유(예: 임차인의 계약 위반, 임대인의 귀책사유 없음 등)를 증명하면 임차인은 보증금 반환청구권을 행사할 수 있습니다.
   - 단, 해지 통지 기간(보통 3개월) 내에 반환되지 않으면 추가 소송 가능.

### 2. **보증금 반환채권 보전 조치 (민법 제323조)**
   - 임대인이

2️⃣ 원본 데이터 학습
임차인이 보증금을 돌려받지 못할 경우, 다음과 같은 방법을 고려해 보세요.

3️⃣ 정제 데이터 학습
임차인이 보증금을 돌려받지 못할 경우, 임차인은 임대인에게 보증금 반환을 요구할 수 있습니다. 임대인이 보증금을 반환하지 않을 경우, 임차인은 법원에 소송을 제기하여 보증금 반환을 청구할 수 있습니다. 또한, 임대인이 보증금을 반환하지 않는 경우, 임차인은 임대차 계약의 해지를 요구할 수 있으며, 이 경우 임대인은 보증금을 반환해야 합니다. 임차인은 임대차 계약의 내용을 확인하고, 필요한 경우 법률 전문가의 상담을 받는 것이 좋습니다.



---
## 4️⃣ LLM-as-Judge 채점

GPT-4o-mini가 4가지 기준 × 10점 = **총 40점**으로 평가

In [31]:
def judge(question, answer):
    prompt = f"""당신은 대한민국 법률 전문가입니다.

[질문]
{question}

[답변]
{answer[:800]}

4가지 기준 각 1~10점. JSON만 출력:
{{"법률_정확성": N, "핵심_포함": N, "오류_여부": N, "실용성": N, "총점": N}}"""

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        if "```" in text:
            text = text.split("```")[1].replace("json", "").strip()
        return json.loads(text)
    except:
        return None

# 5개 질문 × 3개 모델 = 15번 채점
judge_results = []
print("🧑‍⚖️ Judge 채점 중...\n")

for i, q in enumerate(TEST_QUESTIONS, 1):
    print(f"  [{i}/5] {q[:30]}...")
    j_base = judge(q, baseline_results[i-1]["baseline_answer"])
    j_raw = judge(q, finetuned_results[i-1]["finetuned_raw"])
    j_ref = judge(q, refined_answers[i-1])
    judge_results.append({
        "question": q, "base": j_base, "raw": j_raw, "refined": j_ref,
    })
    time.sleep(0.5)

print("\n✅ 채점 완료")

🧑‍⚖️ Judge 채점 중...

  [1/5] 임차인이 보증금을 돌려받지 못할 때 어떻게 해야 하나요...
  [2/5] 근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해...
  [3/5] 온라인 쇼핑에서 환불을 거부당했을 때 소비자 권리는 무...
  [4/5] 교통사고 후 보험사와 합의할 때 주의사항은 무엇인가요?...
  [5/5] 이웃 간 소음 분쟁이 발생했을 때 법적 대응 방법은 무...

✅ 채점 완료


---
## 5️⃣ 최종 비교표

In [32]:
# 평균 점수 계산
def 평균(key):
    vals = [r[key]["총점"] for r in judge_results if r[key]]
    return sum(vals) / len(vals) if vals else 0

base_avg = 평균("base")
raw_avg = 평균("raw")
refined_avg = 평균("refined")

print("=" * 55)
print("📊 최종 비교 (총점 40 만점)")
print("=" * 55)
print(f"1️⃣ 베이스라인:      {base_avg:.1f} / 40")
print(f"2️⃣ 원본 데이터 학습: {raw_avg:.1f} / 40  ({raw_avg-base_avg:+.1f})")
print(f"3️⃣ 정제 데이터 학습: {refined_avg:.1f} / 40  ({refined_avg-base_avg:+.1f})")
print()

if refined_avg > raw_avg + 1:
    print(f"✅ 정제 학습이 원본 학습보다 {refined_avg-raw_avg:+.1f}점 우수!")
    print("   → 데이터 품질이 모델 성능을 결정한다는 핵심 교훈 확인")
elif raw_avg > refined_avg + 1:
    print(f"🔵 원본 학습이 유리 → 정제 데이터 품질/양 보강 필요")
else:
    print(f"🟡 비슷한 수준 → 더 많은 데이터 또는 에폭 필요")

📊 최종 비교 (총점 40 만점)
1️⃣ 베이스라인:      38.0 / 40
2️⃣ 원본 데이터 학습: 32.0 / 40  (-6.0)
3️⃣ 정제 데이터 학습: 27.8 / 40  (-10.2)

🔵 원본 학습이 유리 → 정제 데이터 품질/양 보강 필요


In [33]:
# 결과 저장
comparison = {
    "baseline_avg": base_avg,
    "raw_trained_avg": raw_avg,
    "refined_trained_avg": refined_avg,
    "details": [
        {
            "question": r["question"],
            "base_score": r["base"]["총점"] if r["base"] else 0,
            "raw_score": r["raw"]["총점"] if r["raw"] else 0,
            "refined_score": r["refined"]["총점"] if r["refined"] else 0,
        }
        for r in judge_results
    ],
}

with open("./outputs/comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(comparison, f, ensure_ascii=False, indent=2)

# 어댑터 저장
trainer.save_model("./exaone4-law-refined")
tokenizer.save_pretrained("./exaone4-law-refined")

print("💾 저장 완료:")
print("  ./exaone4-law-refined/ (정제 학습 어댑터)")
print("  ./outputs/comparison_results.json (비교 결과)")

💾 저장 완료:
  ./exaone4-law-refined/ (정제 학습 어댑터)
  ./outputs/comparison_results.json (비교 결과)
